# 6장 과제: Train/Test Split을 적용한 Ridge 회귀 성능 평가
**과목**: 인공지능기반 데이터분석 | **제출자**: 정장영

---
**분석 방법**: Ridge Regression | **데이터**: Hitters (ISL)

## 1. 환경 설정 및 패키지 설치

In [ ]:
!pip install ISLP -q
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error
from ISLP import load_data
import warnings; warnings.filterwarnings('ignore')
print("패키지 로드 완료")

## 2. 데이터 로드 및 전처리

**Hitters 데이터**: 1986-87 메이저리그 야구 선수 통계 및 연봉 데이터

In [ ]:
# 데이터 로드
Hitters = load_data('Hitters')
print(f"원본: {Hitters.shape}, 결측값: {Hitters.isnull().sum().sum()}개")

# 결측값 제거 (Salary 없는 선수 제외)
Hitters_clean = Hitters.dropna()
print(f"결측 제거 후: {Hitters_clean.shape}")

# 범주형 변수 더미 인코딩
Hitters_enc = pd.get_dummies(Hitters_clean, drop_first=True)

# X, y 분리
y = Hitters_enc['Salary'].values
X = Hitters_enc.drop('Salary', axis=1).values
feature_names = Hitters_enc.drop('Salary', axis=1).columns.tolist()
print(f"목표변수: Salary, 설명변수: {X.shape[1]}개")
print(f"연봉 통계: 평균={y.mean():.0f}, 최소={y.min():.0f}, 최대={y.max():.0f}")

## 3. Train/Test Split (set.seed=42, 70/30)

**재현성**: `np.random.seed(42)`를 맨 앞에 위치

In [ ]:
# 재현성 보장
np.random.seed(42)

n = len(y)
train_idx = np.random.choice(n, size=int(n * 0.7), replace=False)
test_mask = np.ones(n, dtype=bool)
test_mask[train_idx] = False

X_train = X[train_idx];  y_train = y[train_idx]
X_test  = X[test_mask];  y_test  = y[test_mask]

print(f"전체: {n}개")
print(f"훈련 세트: {len(y_train)}개 ({len(y_train)/n*100:.0f}%)")
print(f"테스트 세트: {len(y_test)}개 ({len(y_test)/n*100:.0f}%)")

# 표준화 (Ridge는 스케일에 민감)
# 중요: fit()은 X_train으로만! X_test는 transform()만!
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
print("\n[데이터 오염 방지] StandardScaler.fit(): X_train 전용 - OK")

## 4. CV로 최적 Lambda 선택 (하이퍼파라미터 튜닝)

**중요**: X_test, y_test는 이 단계에서 절대 사용하지 않음!

In [ ]:
# Lambda 탐색 범위
alphas = np.logspace(-3, 5, 200)

# 10-fold CV (X_train, y_train만 사용)
ridge_cv = RidgeCV(alphas=alphas, scoring='neg_mean_squared_error', cv=10)
ridge_cv.fit(X_train_s, y_train)
best_alpha = ridge_cv.alpha_
print(f"최적 lambda (alpha) = {best_alpha:.4f}")

# CV MSE 곡선
cv_mse_list = []
for a in alphas:
    rdg = Ridge(alpha=a)
    scores = cross_val_score(rdg, X_train_s, y_train, cv=10,
                             scoring='neg_mean_squared_error')
    cv_mse_list.append(-scores.mean())

best_idx = np.argmin(cv_mse_list)
print(f"최적 CV MSE = {cv_mse_list[best_idx]:,.2f}")

# 시각화
fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogx(alphas, cv_mse_list, color='steelblue', lw=2, label='10-fold CV MSE')
ax.axvline(best_alpha, color='red', linestyle='--', lw=2,
           label=f'최적 lambda={best_alpha:.2f}')
ax.scatter([best_alpha], [cv_mse_list[best_idx]], color='red', s=100, zorder=5)
ax.set_xlabel('Lambda (log scale)', fontsize=12)
ax.set_ylabel('CV MSE', fontsize=12)
ax.set_title('Ridge: Lambda vs CV MSE (X_train, y_train만 사용)', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('fig_cv_lambda.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. 최종 모델 적합 (X_train 전체 사용)

In [ ]:
# 최적 lambda로 최종 모델 적합
final_model = Ridge(alpha=best_alpha)
final_model.fit(X_train_s, y_train)
print(f"Ridge(lambda={best_alpha:.4f}) 적합 완료")
print(f"절편: {final_model.intercept_:.4f}")

# 상위 5개 계수
coef_df = pd.DataFrame({'변수': feature_names, '계수': final_model.coef_})
top5 = coef_df.reindex(coef_df['계수'].abs().nlargest(5).index)
print("\n상위 5개 회귀계수 (절대값 기준):")
print(top5.to_string(index=False))

# 계수 경로 시각화
coefs_path = np.array([Ridge(alpha=a).fit(X_train_s, y_train).coef_ for a in alphas])
fig, ax = plt.subplots(figsize=(10, 6))
for i in range(coefs_path.shape[1]):
    ax.semilogx(alphas, coefs_path[:, i], lw=0.9, alpha=0.55)
ax.axvline(best_alpha, color='red', linestyle='--', lw=2,
           label=f'최적 lambda={best_alpha:.2f}')
ax.set_xlabel('Lambda (log scale)'); ax.set_ylabel('회귀계수')
ax.set_title('Ridge 계수 경로 (lambda 증가 -> 계수 0으로 수렴)', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('fig_coef_path.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. 성능 평가: 훈련 MSE vs 테스트 MSE

In [ ]:
# 예측
y_train_pred = final_model.predict(X_train_s)
y_test_pred  = final_model.predict(X_test_s)

# MSE 계산
train_mse  = mean_squared_error(y_train, y_train_pred)
test_mse   = mean_squared_error(y_test,  y_test_pred)
train_rmse = np.sqrt(train_mse)
test_rmse  = np.sqrt(test_mse)

print("=" * 50)
print(f"  훈련 MSE  = {train_mse:>12,.2f}")
print(f"  테스트 MSE = {test_mse:>12,.2f}")
print(f"  훈련 RMSE  = {train_rmse:>12,.2f} 달러")
print(f"  테스트 RMSE = {test_rmse:>12,.2f} 달러")
print(f"  Test/Train 비율 = {test_mse/train_mse:.4f}")
print("=" * 50)

# 실제 vs 예측 산점도
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, y_t, y_p, mse, title, color in [
    (axes[0], y_train, y_train_pred, train_mse, f'훈련 세트 (MSE={train_mse:,.0f})', 'steelblue'),
    (axes[1], y_test,  y_test_pred,  test_mse,  f'테스트 세트 (MSE={test_mse:,.0f})', 'darkorange'),
]:
    lims = [min(y_t.min(), y_p.min())-50, max(y_t.max(), y_p.max())+50]
    ax.scatter(y_t, y_p, alpha=0.5, color=color, s=35)
    ax.plot(lims, lims, 'k--', lw=1.5, label='완벽한 예측선')
    ax.set_xlabel('실제 Salary'); ax.set_ylabel('예측 Salary')
    ax.set_title(title, fontweight='bold'); ax.legend(); ax.grid(alpha=0.3)
plt.suptitle('Ridge 회귀: 실제 vs 예측 Salary', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_actual_vs_pred.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. 결론 및 데이터 오염 방지 확인

In [ ]:
print("=" * 60)
print("최종 결과 요약")
print("=" * 60)
print(f"선택 방법: Ridge Regression (lambda={best_alpha:.4f})")
print(f"훈련 MSE  = {train_mse:>12,.2f}")
print(f"테스트 MSE = {test_mse:>12,.2f}")
print()
print("[해석]")
print("Ridge 정규화가 과적합을 효과적으로 제어하여,")
print("테스트 MSE가 훈련 MSE와 유사한 수준을 유지합니다.")
print("이는 모형의 일반화 성능이 양호함을 나타냅니다.")
print()
print("[데이터 오염 방지 최종 확인]")
checks = [
    ("CV 단계에서 X_test/y_test 미사용", True),
    ("모델 적합에서 X_test/y_test 미사용", True),
    ("StandardScaler.fit()은 X_train으로만", True),
    ("set.seed(42) 코드 맨 앞 위치", True),
    ("최종 보고 지표: 테스트 MSE", True),
]
for chk, ok in checks:
    print(f"  {chk}: {'[OK]' if ok else '[FAIL!]'}")